# Nyaya — best reader on the best retriever

Two changes each moved Eval-v1 with a confidence interval clear of zero, measured one at a time
against `base-768` (Qwen2.5-3B, zero-shot e5-base retriever, 768 tokens, k=8):

- retriever: `nyaya-embed-v1` in the dense stage, +3.9 points (`base-768-embed-v1`, 39.7%);
- reader: `Qwen/Qwen3-4B-Instruct-2507`, +14.8 points (`qwen3-4b`, 50.6%).

This run combines them and pairs the result against both parents. Whatever it scores becomes the
number the README quotes for the system, with its interval.

**Settings:** GPU T4 x2, Internet On, Input: `jitendrajha98/nyaya-model-src`. ~4 h (the 4B
reader writes ~300-word answers at ~31 s per question on a T4).


In [ ]:
# --- setup -------------------------------------------------------------
import glob, os, shutil, subprocess, sys, time

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["WANDB_DISABLED"] = "true"

cands = glob.glob("/kaggle/input/nyaya-model-src/**/pyproject.toml", recursive=True)
if not cands:
    import kagglehub
    root = kagglehub.dataset_download("jitendrajha98/nyaya-model-src")
    cands = glob.glob(os.path.join(root, "**", "pyproject.toml"), recursive=True)
assert cands, "dataset jitendrajha98/nyaya-model-src not available to this kernel"
SRC = os.path.dirname(cands[0])
WORK = "/kaggle/working/nyaya-model"
shutil.copytree(SRC, WORK, dirs_exist_ok=True)
os.chdir(WORK)
sys.path.insert(0, "src")

PROGRESS = "/kaggle/working/progress.txt"


def note(msg: str) -> None:
    print(msg, flush=True)
    with open(PROGRESS, "a", encoding="utf-8") as fh:
        fh.write(time.strftime("%H:%M:%S ") + msg + chr(10))


def run(cmd):
    note("$ " + " ".join(cmd))
    proc = subprocess.run(cmd, text=True)
    if proc.returncode != 0:
        note(f"FAILED exit {proc.returncode}")
        raise RuntimeError(f"step failed (exit {proc.returncode}): {' '.join(cmd)}")
    note("ok")


note(f"setup: source={SRC}")
run([sys.executable, "-m", "pip", "-q", "install", "-r", "requirements-train.txt"])


In [ ]:
# --- Eval-v1 + GPU preflight -------------------------------------------
import torch

run([sys.executable, "scripts/25_build_eval_v1.py"])
if not torch.cuda.is_available():
    raise RuntimeError("No GPU. Settings -> Accelerator -> GPU T4 x2.")
major, minor = torch.cuda.get_device_capability(0)
note(f"preflight: {torch.cuda.get_device_name(0)} sm_{major}{minor}; torch {torch.__version__}")
for parent in ("qwen3-4b", "base-768-embed-v1"):
    assert os.path.exists(f"outputs/eval-v1/{parent}/predictions.jsonl"), f"{parent} predictions missing from the snapshot"


In [ ]:
# --- the run: Qwen3-4B reader, embed-v1 retriever --------------------------
READER = "Qwen/Qwen3-4B-Instruct-2507"
LABEL = "qwen3-4b-embed-v1"
# --dense-model is explicit even though embed-v1 is the default now, so this cell means the
# same thing whichever snapshot it runs on.
COMMON = ["--model", READER, "--adapter", "none", "--split", "all", "--dense",
          "--dense-model", "NyayaLabs98/nyaya-embed-v1", "--k", "8",
          "--max-new-tokens", "768", "--batch-size", "2"]
run([sys.executable, "scripts/26_eval_v1_run.py", "--limit", "4", "--label", "smoke", *COMMON])
t0 = time.time()
run([sys.executable, "scripts/26_eval_v1_run.py", "--label", LABEL, *COMMON])
note(f"{LABEL} done in {(time.time() - t0) / 60:.0f} min")
run([sys.executable, "scripts/27_compare_runs.py", "--a", "qwen3-4b", "--b", LABEL])
shutil.copy(f"reports/eval_v1_comparison_{LABEL}.json", f"/kaggle/working/eval_v1_comparison_{LABEL}_vs_qwen3-4b.json")
run([sys.executable, "scripts/27_compare_runs.py", "--a", "base-768-embed-v1", "--b", LABEL])
shutil.copy(f"reports/eval_v1_comparison_{LABEL}.json", f"/kaggle/working/eval_v1_comparison_{LABEL}_vs_base-768-embed-v1.json")


In [ ]:
# --- results: download these --------------------------------------------
import json, pathlib

out = pathlib.Path("/kaggle/working/nyaya-qwen3-embed")
out.mkdir(exist_ok=True)
results = json.load(open("reports/eval_v1_results.json", encoding="utf-8"))
for label in ("base-768", "base-768-embed-v1", "qwen3-4b", LABEL):
    m = results[label]["metrics"]
    note(f"{label:<20} fact_recall {m['fact_recall']:.1%}  citation {m['citation_accuracy']:.1%}  all_facts {m['all_facts_accuracy']:.1%}")
shutil.copy("reports/eval_v1_results.json", out)
for f in glob.glob("/kaggle/working/eval_v1_comparison_*_vs_*.json"):
    shutil.copy(f, out)
shutil.copy(f"outputs/eval-v1/{LABEL}/predictions.jsonl", out / f"{LABEL}_predictions.jsonl")
shutil.make_archive("/kaggle/working/nyaya-qwen3-embed", "zip", out)
note("collected: " + ", ".join(sorted(p.name for p in out.iterdir())))
